# 3.7. Weight Decay
D2L의 Weight Decay장을 PyTorch 기준으로 정리함.

앞장에서 overfitting은 training data를 외워버릴 수 있다고 했다.
weight Decay는 이런 overfitting을 줄이기 위해 사용한다. 

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [ ]:
%matplotlib inline
import random
import torch
import matplotlib.pyplot as plt
from torch.distributions.multinomial import Multinomial
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

torch.manual_seed(0)
random.seed(0)

print("PyTorch version:", torch.__version__)

## 1. 기존 선형회귀 loss

선형회귀 모델은 다음과 같다.

$$
\hat{y} = w^T x + b
$$

여기서:

* $x$ : 입력 feature
* $w$ : weight, 가중치
* $b$ : bias, 편향
* $\hat{y}$ : 모델 예측값
* $y$ : 실제 정답 label

기존 선형회귀의 loss는 다음과 같다.

$$
L(w, b) =
\frac{1}{n}
\sum_{i=1}^{n}
\frac{1}{2}
\left(w^T x^{(i)} + b - y^{(i)}\right)^2
$$

이 loss는 모델 예측값과 정답의 차이를 줄이는 역할을 한다.
예측값 $\hat{y}$가 정답 $y$와 비슷해지도록 $w$, $b$를 학습한다.

## 2. overfitting은 왜 생기는가

training data는 적은데 feature가 많을 때 생긴다.

예를 들어서 feature가 200개인데, training data가 20개라고 했을때
$$
w_1x_1 + w_2x_2 + \cdots + w_{200}x_{200} + b
$$
여기에서 모델은 200개의 weight를 조정할 수 있다. 근데 training data가 20개 밖에 없으니 모델 입장에서는 자유도가 너무 많다.

이런 상황에서는 모델이 실제 규칙을 배우는게 아니고, training data 20개에만 맞는 이상한 weight 조합을 찾을 수 있다. 이런걸 overfitting이라고 한다.

## 3. Weight Decay의 핵심 아이디어

Weight decay는 기존 loss에 weight 크기 벌점을 추가한다.

기존 loss는 다음과 같았다.

$$
L(w, b)
$$

Weight decay를 적용하면 다음처럼 바뀐다.

$$
L(w, b) + \frac{\lambda}{2}|w|^2
$$

여기서 새로 추가된 부분은 다음이다.

$$
\frac{\lambda}{2}|w|^2
$$

이 항은 weight가 커질수록 loss를 증가시킨다.

정답을 맞추는 것도 중요하지만, weight를 너무 크게 만들면 안좋게 하는 것이다.

## 4. $|w|^2$는 무엇인가?

$w$가 다음과 같다고 해보자.

$$
w =
\begin{bmatrix}
w_1 \
w_2 \
w_3 \
\vdots \
w_d
\end{bmatrix}
$$

그러면 $|w|^2$는 다음과 같다.
$$
w_1^2 + w_2^2 + w_3^2 + \cdots + w_d^2
$$
 
모든 weight를 제곱해서 더한 값이다. D2L에서는 이것을 다음처럼 사용한다.  
$$
\frac{1}{2}
\left(
w_1^2 + w_2^2 + \cdots + w_d^2
\right)
$$

PyTorch로 쓰면 다음과 같다.
```py
def l2_penalty(w):
    return (w ** 2).sum() / 2
```

## 5. 왜 weight가 크면 문제가 되는가?

선형회귀 모델을 다시 보자.
$$
w_1x_1 + w_2x_2 + \cdots + w_dx_d + b
$$

여기서 어떤 weight가 너무 크면, 그 feature가 조금만 변해도 예측값이 크게 변한다.

예를 들어서 $w_3$가 매우 크다고 해보자.

$$
w_3 = 100
$$

그러면 $x_3$가 조금만 변해도 예측값이 크게 변한다. 모델이 특정 feature에 너무 민감하다는 뜻이다.

training data 안에서는 잘 맞을 수 있다. 그런데 새로운 data에서는 예측이 쉽게 망가질 수 있다. 그래서 weight decay는 weight가 너무 커지는 것을 막는다.

## 6. $\lambda$는 무엇인가?

Weight decay가 적용된 loss는 다음과 같다.

$$
L(w, b) + \frac{\lambda}{2}|w|^2
$$

여기서 $\lambda$는 regularization strength이다. weight 크기에 대한 벌점을 얼마나 강하게 줄지 정하는 값이다.

$\lambda = 0$인 경우
$$
L(w, b)
$$

이 경우에 weight decay가 없는 거랑 같다.

$\lambda$가 작은 경우
weight 크기에 대한 벌점이 약하다.
모델이 weight를 비교적 자유롭게 키울 수 있다.

$\lambda$가 큰 경우
weight 크기에 대한 벌점이 강하다.
모델이 weight를 크게 만들기 어렵다.

하지만 $\lambda$가 너무 크면 문제가 생긴다.
모델이 weight를 거의 사용하지 못해서 underfitting이 생길 수 있다.

정리하면 다음과 같다.

$$
\lambda \text{가 너무 작음}
\Rightarrow
\text{overfitting을 못 막을 수 있음}
$$

$$
\lambda \text{가 너무 큼}
\Rightarrow
\text{underfitting이 생길 수 있음}
$$

## 7. 왜 이름이 Weight Decay인가?

Weight decay가 들어가면 weight update 식은 다음과 비슷해진다.
$$
\eta \nabla_w L(w, b)
$$

여기서:

$\eta$ : learning rate  
$\lambda$ : weight decay strength  
$\nabla_w L(w, b)$ : 기존 loss에 대한 gradient  

$$
(1 - \eta \lambda)w
$$

이 항은 기존 weight를 조금 줄이는 역할을 한다.
예를 들어 다음과 같다고 하자.

$$
\eta = 0.01
$$

$$
\lambda = 3
$$

그러면:
$$
1 - 0.01 \times 3 = 0.97
$$

따라서 weight는 매 step마다 이런 식으로 줄어드는 압력을 받는다.

$$
w \leftarrow 0.97w
$$

그래서 이름이 weight decay다.

weight가 decay, 즉 조금씩 감소하는 방향으로 압력을 받기 때문이다.

weight decay는 weight를 무조건 0으로 만드는 것이 아니다.
$$
\text{예측 오차를 줄이는 힘}
+
\text{weight를 작게 만드는 힘}
$$

그래서 필요한 weight는 남고, 커지는 weight는 억제된다.

## 8. L2 Regularization

Weight decay에서 자주 사용하는 방식은 L2 regularization이다.

L2 penalty는 다음과 같다.
$$
w_1^2 + w_2^2 + \cdots + w_d^2
$$

loss에 추가하면 다음과 같다.

$$
L(w, b) + \frac{\lambda}{2}\left(w_1^2 + w_2^2 + \cdots + w_d^2\right)
$$

L2 regularization은 weight를 제곱하기 때문에 큰 weight를 강하게 벌준다.  
예를 들어서:

$$
2^2 = 4
$$

$$
10^2 = 100
$$

weight가 커질수록 패널티가 빠르게 커진다.  
L2 regularization은 weight가 너무 커지지 못하게 막는다.